# IEEE 39-bus fault localization (day 1)

Generates the fault dataset, builds PyG graphs, and trains the flattened MLP baseline. GNN training is day 2.

Design: [docs/GRID_FAULTS.md](../docs/GRID_FAULTS.md).

In [ ]:
from pathlib import Path
import json
import numpy as np

from grid.simulate import generate_and_save
from grid.graph import (
    build_case39_topology,
    plot_dv_sample,
    save_processed_dataset,
    samples_to_arrays,
    stratified_split,
    zscore_fit,
    zscore_transform,
)
from grid.baseline import train_mlp_baseline, save_baseline_metrics

OUT = Path("../results")
SEED = 0
SCENARIOS = 50

In [ ]:
samples, bus_ids, method_counts = generate_and_save(OUT, n_scenarios=SCENARIOS, seed=SEED)
print(len(samples), "graphs; voltage methods:", method_counts)

In [ ]:
topology = build_case39_topology()
x_raw, labels, _ = samples_to_arrays(samples)
train_idx, val_idx, test_idx = stratified_split(labels, seed=SEED)
mean, std = zscore_fit(x_raw[train_idx])
x_norm = zscore_transform(x_raw, mean, std)
methods = np.array([s.voltage_method for s in samples], dtype=object)
save_processed_dataset(
    OUT, x_raw, x_norm, labels, topology,
    train_idx, val_idx, test_idx, mean, std, voltage_methods=methods,
)
plot_dv_sample(samples[0], topology, OUT / "fault_dv_sample.png")

In [ ]:
metrics = train_mlp_baseline(
    x_norm, labels, train_idx, val_idx, test_idx, x_raw, seed=SEED
)
save_baseline_metrics(metrics, OUT / "mlp_baseline.json")
print(metrics)

## Results (seed 0, full 1,950-graph run)

| Model | Test accuracy | Top-3 |
|-------|---------------|-------|
| MLP | 0.956 | 0.982 |
| Min during-fault V | 1.000 | 1.000 |

Voltage method: **ybus_bolted** for all samples.